In [2]:
import pandas as pd
import numpy as np

# Mostrar todas las columnas sin truncar
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', None)

In [5]:
RUTA_VIVIENDA = '../data/raw/Datos de la vivienda.csv'
RUTA_SERVICIOS_HOGAR = '../data/raw/Servicios del hogar.csv'
RUTA_CONDICIONES_VIDA = '../data/raw/Condiciones de vida del hogar y tenencia de bienes.csv'

df_vivienda = pd.read_csv(RUTA_VIVIENDA, sep=';', dtype=str, encoding='utf-8-sig')
df_servicios_hogar = pd.read_csv(RUTA_SERVICIOS_HOGAR, sep=';', dtype=str, encoding='utf-8-sig')
df_condiciones_vida = pd.read_csv(RUTA_CONDICIONES_VIDA, sep=';', dtype=str, encoding='utf-8-sig')

In [6]:
print('Vivienda:', df_vivienda.shape)
print('Servicios del hogar:', df_servicios_hogar.shape)
print('Condiciones de vida:', df_condiciones_vida.shape)

Vivienda: (86848, 46)
Servicios del hogar: (87060, 98)
Condiciones de vida: (87060, 145)


In [7]:
print('DIRECTORIO unicos en vivienda:', df_vivienda['DIRECTORIO'].nunique())
print('Filas totales en vivienda:', len(df_vivienda))
print('Duplicados de DIRECTORIO en vivienda:', df_vivienda['DIRECTORIO'].duplicated().sum())

DIRECTORIO unicos en vivienda: 86848
Filas totales en vivienda: 86848
Duplicados de DIRECTORIO en vivienda: 0


In [8]:
for nombre, df in [('servicios_hogar', df_servicios_hogar), ('condiciones_vida', df_condiciones_vida)]:
    print('---', nombre, '---')
    print('DIRECTORIO unicos:', df['DIRECTORIO'].nunique())
    print('Duplicados de solo DIRECTORIO:', df['DIRECTORIO'].duplicated().sum())
    print('Duplicados de (DIRECTORIO, ORDEN):', df.duplicated(subset=['DIRECTORIO', 'ORDEN']).sum())
    print()

--- servicios_hogar ---
DIRECTORIO unicos: 86848
Duplicados de solo DIRECTORIO: 212
Duplicados de (DIRECTORIO, ORDEN): 0

--- condiciones_vida ---
DIRECTORIO unicos: 86848
Duplicados de solo DIRECTORIO: 212
Duplicados de (DIRECTORIO, ORDEN): 0



In [9]:
# Union condiciones_vida -> servicios_hogar por (DIRECTORIO, ORDEN)
union_1 = df_condiciones_vida.merge(
    df_servicios_hogar,
    on=['DIRECTORIO', 'ORDEN'],
    how='left',
    suffixes=('', '_servicios')
)
print('Filas condiciones_vida original:', len(df_condiciones_vida))
print('Filas tras unir con servicios_hogar:', len(union_1))

# Union del resultado -> vivienda por DIRECTORIO
union_final = union_1.merge(
    df_vivienda,
    on='DIRECTORIO',
    how='left',
    suffixes=('', '_vivienda')
)
print('Filas tras unir con vivienda:', len(union_final))

Filas condiciones_vida original: 87060
Filas tras unir con servicios_hogar: 87060
Filas tras unir con vivienda: 87060


In [10]:
# Verificar que el join no dejo filas huerfanas (sin pareja en las otras tablas)

print('Filas sin match en servicios_hogar:', union_final['I_HOGAR'].isna().sum())
print('Filas sin match en vivienda:', union_final['REGION'].isna().sum())

Filas sin match en servicios_hogar: 0
Filas sin match en vivienda: 0


In [11]:
df_integrado = union_final.copy()

nulos_pct = (df_integrado.isna().sum() / len(df_integrado) * 100)
columnas_100_vacias = nulos_pct[nulos_pct == 100].index.tolist()

print('Total de columnas en la tabla integrada:', df_integrado.shape[1])
print('Columnas 100% vacias:', len(columnas_100_vacias))
print(columnas_100_vacias)

Total de columnas en la tabla integrada: 286
Columnas 100% vacias: 21
['P9025', 'P3180', 'P9005', 'P784', 'P1072', 'P1077', 'P795', 'P1913', 'P3202', 'P3203', 'P3516', 'P1892', 'P5046S1', 'P5012', 'P3169', 'P3172', 'P3174', 'P8520', 'P4065', 'P5661', 'P3157']


In [12]:
df_diccionario = pd.read_csv('../data/reference/Plantilla_Diccionario_Datos.csv', sep=';', dtype=str, encoding='utf-8-sig')

columnas_vacias = ['P9025', 'P3180', 'P9005', 'P784', 'P1072', 'P1077', 'P795', 'P1913',
                   'P3202', 'P3203', 'P3516', 'P1892', 'P5046S1', 'P5012', 'P3169',
                   'P3172', 'P3174', 'P8520', 'P4065', 'P5661', 'P3157']

consulta = df_diccionario[df_diccionario['Nombre de la variable o la columna'].isin(columnas_vacias)]
print(consulta[['Nombre de la variable o la columna', 'Descripción de la variable o la columna']].to_string(index=False))

Nombre de la variable o la columna                                                                                                                                                        Descripción de la variable o la columna
                             P8520                                                                                                  8. ¿Con cuáles de los siguientes servicios públicos, privados o comunales cuenta la vivienda?
                             P4065                                                                                                                                 9. ¿En los últimos 12 MESES, la vivienda ha sido afectada por:
                             P5661                                                      10.  ¿En los últimos 12 MESES, con qué frecuencia se han presentado los siguientes problemas en el sector donde está ubicada su vivienda:
                             P3157                                                              

In [ ]:
df_diccionario = pd.read_csv('../data/reference/Plantilla_Diccionario_Datos.csv', sep=';', dtype=str, encoding='utf-8-sig')

columnas_vacias = ['P9025', 'P3180', 'P9005', 'P784', 'P1072', 'P1077', 'P795', 'P1913',
                   'P3202', 'P3203', 'P3516', 'P1892', 'P5046S1', 'P5012', 'P3169',
                   'P3172', 'P3174', 'P8520', 'P4065', 'P5661', 'P3157']

consulta = df_diccionario[df_diccionario['Nombre de la variable o la columna'].isin(columnas_vacias)]
print(consulta[['Nombre de la variable o la columna', 'Descripción de la variable o la columna']].to_string(index=False))

Nombre de la variable o la columna                                                                                                                                                        Descripción de la variable o la columna
                             P8520                                                                                                  8. ¿Con cuáles de los siguientes servicios públicos, privados o comunales cuenta la vivienda?
                             P4065                                                                                                                                 9. ¿En los últimos 12 MESES, la vivienda ha sido afectada por:
                             P5661                                                      10.  ¿En los últimos 12 MESES, con qué frecuencia se han presentado los siguientes problemas en el sector donde está ubicada su vivienda:
                             P3157                                                              

In [13]:
for col in columnas_vacias:
    hijas = [c for c in df_integrado.columns if c.startswith(col) and c != col]
    if hijas:
        alguna_con_dato = df_integrado[hijas].notna().any().any()
        print(f'{col}: tiene {len(hijas)} columnas hijas, alguna con dato = {alguna_con_dato}')
    else:
        print(f'{col}: NO tiene columnas hijas -- revisar por que esta vacia')

P9025: tiene 2 columnas hijas, alguna con dato = True
P3180: tiene 15 columnas hijas, alguna con dato = True
P9005: tiene 5 columnas hijas, alguna con dato = True
P784: tiene 11 columnas hijas, alguna con dato = True
P1072: tiene 2 columnas hijas, alguna con dato = True
P1077: tiene 25 columnas hijas, alguna con dato = True
P795: tiene 6 columnas hijas, alguna con dato = True
P1913: tiene 12 columnas hijas, alguna con dato = True
P3202: tiene 12 columnas hijas, alguna con dato = True
P3203: tiene 14 columnas hijas, alguna con dato = True
P3516: tiene 8 columnas hijas, alguna con dato = True
P1892: tiene 4 columnas hijas, alguna con dato = True
P5046S1: tiene 7 columnas hijas, alguna con dato = True
P5012: tiene 8 columnas hijas, alguna con dato = True
P3169: tiene 4 columnas hijas, alguna con dato = True
P3172: tiene 4 columnas hijas, alguna con dato = True
P3174: tiene 5 columnas hijas, alguna con dato = True
P8520: tiene 6 columnas hijas, alguna con dato = True
P4065: tiene 5 columna

In [14]:
# Distribucion de nulos en toda la tabla integrada, agrupada en rangos

nulos_pct = (df_integrado.isna().sum() / len(df_integrado) * 100)

rangos = pd.cut(
    nulos_pct,
    bins=[-0.1, 0, 5, 25, 50, 75, 95, 99.9, 100],
    labels=['0%', '0-5%', '5-25%', '25-50%', '50-75%', '75-95%', '95-99.9%', '100%']
)

print(rangos.value_counts().sort_index())

0%          154
0-5%          7
5-25%        12
25-50%       15
50-75%       20
75-95%       15
95-99.9%     38
100%         25
Name: count, dtype: int64


In [15]:
nulos_pct = (df_integrado.isna().sum() / len(df_integrado) * 100)
columnas_100_ahora = nulos_pct[nulos_pct == 100].index.tolist()

columnas_100_antes = ['P9025', 'P3180', 'P9005', 'P784', 'P1072', 'P1077', 'P795', 'P1913',
                       'P3202', 'P3203', 'P3516', 'P1892', 'P5046S1', 'P5012', 'P3169',
                       'P3172', 'P3174', 'P8520', 'P4065', 'P5661', 'P3157']

nuevas = [c for c in columnas_100_ahora if c not in columnas_100_antes]
print('Total 100% vacias ahora:', len(columnas_100_ahora))
print('Columnas nuevas no identificadas antes:', nuevas)

Total 100% vacias ahora: 21
Columnas nuevas no identificadas antes: []


In [19]:
columnas_95_999 = nulos_pct[(nulos_pct >= 95) & (nulos_pct < 100)].index.tolist()
print('Cantidad en el rango 95-99.9%:', len(columnas_95_999))
print(columnas_95_999)

muestra = df_diccionario[df_diccionario['Nombre de la variable o la columna'].isin(columnas_95_999)]
print(muestra[['Nombre de la variable o la columna', 'Descripción de la variable o la columna']].to_string(index=False))

Cantidad en el rango 95-99.9%: 42
['P784S7A1', 'P784S7A2', 'P784S6A1', 'P784S6A2', 'P784S4A1', 'P1077S23A1', 'P795S4', 'P3202S1', 'P3202S2', 'P3202S3', 'P3202S4', 'P3202S5', 'P3202S6', 'P3202S7', 'P3202S8', 'P3202S9', 'P3202S11', 'P3203S1', 'P3203S2', 'P3203S3', 'P3203S4', 'P3203S5', 'P3203S6', 'P3203S7', 'P3203S8', 'P3203S9', 'P3203S10', 'P3203S11', 'P3203S12', 'P3203S13', 'P3203S14', 'P5046S1A7', 'P3168', 'P8534', 'P3172S1', 'P3172S2', 'P3172S3', 'P3174S1', 'P3174S2', 'P3174S3', 'P3174S4', 'P3174S5']
Nombre de la variable o la columna                                                                Descripción de la variable o la columna
                         P5046S1A7                                                                                       7. Medicamentos 
                             P3168 31. ¿Cuánto tiempo gasta caminando para llegar a ese lugar, recoger el agua y regresar a la vivienda? 
                             P8534                               33A. ¿El hoga

In [20]:
variables_nucleo = {
    'C1_territorio': ['REGION', 'Clase', 'P1_DEPARTAMENTO'],
    'C4_servicios_vivienda': ['P8520S5', 'P8520S3'],
    'C6_choques': [f'P3202S{i}' for i in range(1, 13)],
    'C7_estrategias': [f'P3203S{i}' for i in range(1, 15)],
    'C10_ingreso': ['PERCAPITA', 'I_HOGAR', 'CANT_PERSONAS_HOGAR'],
    'medidas_base': ['FEX_C', 'PROB_IAMG', 'PROB_IAG'],
}

for grupo, cols in variables_nucleo.items():
    faltantes = [c for c in cols if c not in df_integrado.columns]
    print(f'{grupo}: {len(cols)} variables, faltantes = {faltantes}')

C1_territorio: 3 variables, faltantes = ['Clase']
C4_servicios_vivienda: 2 variables, faltantes = []
C6_choques: 12 variables, faltantes = []
C7_estrategias: 14 variables, faltantes = []
C10_ingreso: 3 variables, faltantes = []
medidas_base: 3 variables, faltantes = []


In [21]:
print('=== P8520S5 (acueducto) ===')
print(df_integrado['P8520S5'].value_counts(dropna=False))
print()
print('=== P8520S3 (alcantarillado) ===')
print(df_integrado['P8520S3'].value_counts(dropna=False))

=== P8520S5 (acueducto) ===
P8520S5
1    64913
2    22147
Name: count, dtype: int64

=== P8520S3 (alcantarillado) ===
P8520S3
1    43912
2    43148
Name: count, dtype: int64


In [22]:
columnas_candidatas = [c for c in df_integrado.columns if 'clase' in c.lower()]
print('Columnas que contienen "clase":', columnas_candidatas)

Columnas que contienen "clase": ['CLASE']


In [23]:
print('=== REGION ===')
print(df_integrado['REGION'].value_counts(dropna=False))
print()
print('=== CLASE ===')
print(df_integrado['CLASE'].value_counts(dropna=False))
print()
print('=== P1_DEPARTAMENTO ===')
print('Valores unicos:', df_integrado['P1_DEPARTAMENTO'].nunique())
print('Longitudes de caracter:', df_integrado['P1_DEPARTAMENTO'].str.len().value_counts())

=== REGION ===
REGION
1    18333
9    17422
3    16691
2    16483
4     8449
6     3732
7     3195
5     1866
8      889
Name: count, dtype: int64

=== CLASE ===
CLASE
1    44351
2    42709
Name: count, dtype: int64

=== P1_DEPARTAMENTO ===
Valores unicos: 33
Longitudes de caracter: P1_DEPARTAMENTO
2    80695
1     6365
Name: count, dtype: int64


In [24]:
choques = [f'P3202S{i}' for i in range(1, 13)]

conteo_choques = {}
for c in choques:
    conteo_choques[c] = (df_integrado[c] == '1').sum()

conteo_choques = pd.Series(conteo_choques).sort_values(ascending=False)
print(conteo_choques)

P3202S12    74125
P3202S10     5484
P3202S1      3355
P3202S8      1740
P3202S3      1458
P3202S11     1227
P3202S2      1217
P3202S5       875
P3202S4       176
P3202S6       122
P3202S9       110
P3202S7       108
dtype: int64


In [25]:
estrategias = [f'P3203S{i}' for i in range(1, 15)]

conteo_estrategias = {}
for c in estrategias:
    conteo_estrategias[c] = (df_integrado[c] == '1').sum()

conteo_estrategias = pd.Series(conteo_estrategias).sort_values(ascending=False)
print(conteo_estrategias)

P3203S12    4239
P3203S5     3461
P3203S4     3417
P3203S10    2608
P3203S2     2199
P3203S1      660
P3203S3      404
P3203S11     220
P3203S6      211
P3203S14     127
P3203S13      63
P3203S8       39
P3203S9       30
P3203S7       14
dtype: int64


In [26]:
df_integrado['n_choques'] = df_integrado[choques].apply(lambda fila: (fila == '1').sum(), axis=1)

print('Hogares sin ningun choque marcado:', (df_integrado['n_choques'] == 0).sum())
print('Hogares con al menos un choque:', (df_integrado['n_choques'] > 0).sum())
print()
print(df_integrado['n_choques'].value_counts().sort_index())

Hogares sin ningun choque marcado: 0
Hogares con al menos un choque: 87060

n_choques
1     84639
2      2007
3       336
4        66
5         8
6         1
7         1
8         1
11        1
Name: count, dtype: int64


In [27]:
fila = df_diccionario[df_diccionario['Nombre de la variable o la columna'] == 'P3202S12']
print(fila[['Nombre de la variable o la columna', 'Descripción de la variable o la columna']].to_string(index=False))

Nombre de la variable o la columna Descripción de la variable o la columna
                          P3202S12           12. Ninguno de las anteriores


In [28]:
choques_reales = [f'P3202S{i}' for i in range(1, 12)]  # excluye S12 si es "Ninguno"

df_integrado['n_choques'] = df_integrado[choques_reales].apply(lambda fila: (fila == '1').sum(), axis=1)

print('Hogares sin ningun choque (segun conteo de S1 a S11):', (df_integrado['n_choques'] == 0).sum())
print('Hogares con S12 marcado:', (df_integrado['P3202S12'] == '1').sum())
print()
print('Cruce: hogares con S12 marcado que tambien tienen algun choque en S1-S11:')
print(((df_integrado['P3202S12'] == '1') & (df_integrado['n_choques'] > 0)).sum())
print()
print(df_integrado['n_choques'].value_counts().sort_index())

Hogares sin ningun choque (segun conteo de S1 a S11): 74125
Hogares con S12 marcado: 74125

Cruce: hogares con S12 marcado que tambien tienen algun choque en S1-S11:
0

n_choques
0     74125
1     10514
2      2007
3       336
4        66
5         8
6         1
7         1
8         1
11        1
Name: count, dtype: int64


In [29]:
# Buscar si P3203 tiene una opcion tipo "Ninguna"
fila_p3203 = df_diccionario[df_diccionario['Nombre de la variable o la columna'].str.startswith('P3203', na=False)]
print(fila_p3203[['Nombre de la variable o la columna', 'Descripción de la variable o la columna']].to_string(index=False))

Nombre de la variable o la columna                                               Descripción de la variable o la columna
                             P3203                        17. ¿Qué medidas tomaron para hacerles frente a estos eventos?
                           P3203S1                1. Uno o más miembros del hogar que no trabajaban empezaron a trabajar
                           P3203S2                                                2. Adoptaron nuevas fuente de ingreso 
                           P3203S3                         3. Se fueron a vivir con familiares o acogieron a un familiar
                           P3203S4                                                 4. Gastaron parte o todos sus ahorros
                           P3203S5                           5. Se endeudaron o ampliaron el plazo de alguna(s) deuda(s)
                           P3203S6           6. Vendieron algunos bienes o activos (moto, carro, electrodoméstico, lote)
                           P3203

In [30]:
estrategias = [f'P3203S{i}' for i in range(1, 15)]
df_integrado['n_estrategias'] = df_integrado[estrategias].apply(lambda fila: (fila == '1').sum(), axis=1)

print('=== Cruce choque vs estrategia ===')
print('Hogares sin choque (n_choques=0) y sin estrategia (n_estrategias=0):',
      ((df_integrado['n_choques'] == 0) & (df_integrado['n_estrategias'] == 0)).sum())

print('Hogares sin choque pero CON alguna estrategia marcada (inconsistente):',
      ((df_integrado['n_choques'] == 0) & (df_integrado['n_estrategias'] > 0)).sum())

print('Hogares con choque pero SIN ninguna estrategia marcada:',
      ((df_integrado['n_choques'] > 0) & (df_integrado['n_estrategias'] == 0)).sum())

print('Hogares con choque y con alguna estrategia:',
      ((df_integrado['n_choques'] > 0) & (df_integrado['n_estrategias'] > 0)).sum())

print()
print(df_integrado['n_estrategias'].value_counts().sort_index())

=== Cruce choque vs estrategia ===
Hogares sin choque (n_choques=0) y sin estrategia (n_estrategias=0): 74125
Hogares sin choque pero CON alguna estrategia marcada (inconsistente): 0
Hogares con choque pero SIN ninguna estrategia marcada: 0
Hogares con choque y con alguna estrategia: 12935

n_estrategias
0     74125
1      9526
2      2480
3       657
4       170
5        74
6        19
7         6
8         2
13        1
Name: count, dtype: int64


In [31]:
print('=== PERCAPITA ===')
percapita = pd.to_numeric(df_integrado['PERCAPITA'].str.replace(',', '.'), errors='coerce')
print(percapita.describe())
print('Valores nulos:', percapita.isna().sum())
print('Valores en cero:', (percapita == 0).sum())
print()
print('=== CANT_PERSONAS_HOGAR ===')
print(df_integrado['CANT_PERSONAS_HOGAR'].value_counts().sort_index())

=== PERCAPITA ===
count    8.706000e+04
mean     1.191593e+06
std      2.083874e+06
min      0.000000e+00
25%      3.875000e+05
50%      7.000000e+05
75%      1.323333e+06
max      2.550000e+08
Name: PERCAPITA, dtype: float64
Valores nulos: 0
Valores en cero: 681

=== CANT_PERSONAS_HOGAR ===
CANT_PERSONAS_HOGAR
1     19091
10       63
11       16
12        7
13        9
15        3
2     25059
21        1
3     20377
4     13351
5      5752
6      2091
7       815
8       303
9       122
Name: count, dtype: int64


In [32]:
percapita = pd.to_numeric(df_integrado['PERCAPITA'].str.replace(',', '.'), errors='coerce')

print('Percentiles altos:')
for p in [90, 95, 99, 99.5, 99.9, 99.99, 100]:
    print(f'  p{p}: {percapita.quantile(p/100):,.0f}')

print()
print('Cuantos hogares superan 50 millones per capita:', (percapita > 50_000_000).sum())
print('Cuantos hogares superan 20 millones per capita:', (percapita > 20_000_000).sum())
print('Cuantos hogares superan 10 millones per capita:', (percapita > 10_000_000).sum())

Percentiles altos:
  p90: 2,370,833
  p95: 3,625,844
  p99: 8,500,000
  p99.5: 11,651,917
  p99.9: 22,494,100
  p99.99: 41,537,985
  p100: 255,000,000

Cuantos hogares superan 50 millones per capita: 5
Cuantos hogares superan 20 millones per capita: 127
Cuantos hogares superan 10 millones per capita: 610


In [33]:
i_hogar = pd.to_numeric(df_integrado['I_HOGAR'].str.replace(',', '.'), errors='coerce')

hogares_percapita_cero = df_integrado[percapita == 0]
print('De los 681 hogares con PERCAPITA=0, cuantos tienen I_HOGAR=0 tambien:')
print((i_hogar[percapita == 0] == 0).sum())
print()
print('Distribucion de I_HOGAR en esos 681 hogares:')
print(i_hogar[percapita == 0].describe())

De los 681 hogares con PERCAPITA=0, cuantos tienen I_HOGAR=0 tambien:
678

Distribucion de I_HOGAR en esos 681 hogares:
count    6.810000e+02
mean     7.134116e+03
std      1.170171e+05
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      2.608333e+06
Name: I_HOGAR, dtype: float64


In [34]:
percapita = pd.to_numeric(df_integrado['PERCAPITA'].str.replace(',', '.'), errors='coerce')
i_hogar = pd.to_numeric(df_integrado['I_HOGAR'].str.replace(',', '.'), errors='coerce')

extremos = df_integrado[percapita > 50_000_000][['DIRECTORIO', 'ORDEN', 'CANT_PERSONAS_HOGAR']].copy()
extremos['PERCAPITA'] = percapita[percapita > 50_000_000]
extremos['I_HOGAR'] = i_hogar[percapita > 50_000_000]
print(extremos.to_string(index=False))

DIRECTORIO ORDEN CANT_PERSONAS_HOGAR    PERCAPITA     I_HOGAR
   8393001     1                   1  98228500.00  98228500.0
   8398884     1                   2  93061666.67 186123333.3
   8404352     1                   1  83000000.00  83000000.0
   8457583     1                   1 255000000.00 255000000.0
   8463596     1                   1  57300000.00  57300000.0


In [35]:
casos_raros = df_integrado[(percapita == 0) & (i_hogar != 0)][['DIRECTORIO', 'ORDEN', 'CANT_PERSONAS_HOGAR']].copy()
casos_raros['PERCAPITA'] = percapita[(percapita == 0) & (i_hogar != 0)]
casos_raros['I_HOGAR'] = i_hogar[(percapita == 0) & (i_hogar != 0)]
print(casos_raros.to_string(index=False))

DIRECTORIO ORDEN CANT_PERSONAS_HOGAR  PERCAPITA     I_HOGAR
   8417551     1                   4        0.0 2608333.333
   8448706     1                   3        0.0 1200000.000
   8458761     1                   4        0.0 1050000.000


In [36]:
fex_c = pd.to_numeric(df_integrado['FEX_C'].str.replace(',', '.'), errors='coerce')
prob_iamg = pd.to_numeric(df_integrado['PROB_IAMG'].str.replace(',', '.'), errors='coerce')
prob_iag = pd.to_numeric(df_integrado['PROB_IAG'].str.replace(',', '.'), errors='coerce')

print('=== FEX_C ===')
print(fex_c.describe())
print('Nulos:', fex_c.isna().sum())
print()
print('=== PROB_IAMG ===')
print(prob_iamg.describe())
print('Nulos:', prob_iamg.isna().sum())
print()
print('=== PROB_IAG ===')
print(prob_iag.describe())
print('Nulos:', prob_iag.isna().sum())

=== FEX_C ===
count    87060.000000
mean       217.692285
std        364.067055
min          1.545446
25%         37.191299
50%        104.132269
75%        232.761851
max       4843.503512
Name: FEX_C, dtype: float64
Nulos: 0

=== PROB_IAMG ===
count    86342.000000
mean        25.005813
std         36.301995
min          0.000000
25%          0.000000
50%          3.146038
75%         37.506476
max         99.824798
Name: PROB_IAMG, dtype: float64
Nulos: 718

=== PROB_IAG ===
count    86342.000000
mean         4.243090
std         14.809875
min          0.000000
25%          0.000000
50%          0.000080
75%          0.002130
max         75.222899
Name: PROB_IAG, dtype: float64
Nulos: 718


In [37]:
preguntas_fies = [f'P3516S{i}' for i in range(1, 9)]

tiene_no_informa = (df_integrado[preguntas_fies] == '3').any(axis=1)

print('Hogares con algun "3" (no sabe/no informa) en FIES:', tiene_no_informa.sum())
print('Hogares con PROB_IAMG nulo:', prob_iamg.isna().sum())
print()
print('Coinciden exactamente:', (tiene_no_informa == prob_iamg.isna()).all())

Hogares con algun "3" (no sabe/no informa) en FIES: 718
Hogares con PROB_IAMG nulo: 718

Coinciden exactamente: True


In [38]:
validos = prob_iamg.notna()

prevalencia_iamg = (fex_c[validos] * prob_iamg[validos] / 100).sum() / fex_c[validos].sum() * 100
prevalencia_iag = (fex_c[validos] * prob_iag[validos] / 100).sum() / fex_c[validos].sum() * 100

print(f'Prevalencia ponderada IA moderada/grave: {prevalencia_iamg:.2f}%')
print(f'Prevalencia ponderada IA grave: {prevalencia_iag:.2f}%')
print()
print('Cifra oficial DANE 2025 (moderada/grave): 21.1%')

Prevalencia ponderada IA moderada/grave: 21.10%
Prevalencia ponderada IA grave: 3.42%

Cifra oficial DANE 2025 (moderada/grave): 21.1%


In [39]:
columnas_vacias = ['P9025', 'P3180', 'P9005', 'P784', 'P1072', 'P1077', 'P795', 'P1913',
                   'P3202', 'P3203', 'P3516', 'P1892', 'P5046S1', 'P5012', 'P3169',
                   'P3172', 'P3174', 'P8520', 'P4065', 'P5661', 'P3157']

for col in columnas_vacias:
    en_vivienda = col in df_vivienda.columns
    en_servicios = col in df_servicios_hogar.columns
    en_condiciones = col in df_condiciones_vida.columns
    print(f'{col:<10} vivienda={en_vivienda}  servicios_hogar={en_servicios}  condiciones_vida={en_condiciones}')

P9025      vivienda=False  servicios_hogar=False  condiciones_vida=True
P3180      vivienda=False  servicios_hogar=False  condiciones_vida=True
P9005      vivienda=False  servicios_hogar=False  condiciones_vida=True
P784       vivienda=False  servicios_hogar=False  condiciones_vida=True
P1072      vivienda=False  servicios_hogar=False  condiciones_vida=True
P1077      vivienda=False  servicios_hogar=False  condiciones_vida=True
P795       vivienda=False  servicios_hogar=False  condiciones_vida=True
P1913      vivienda=False  servicios_hogar=False  condiciones_vida=True
P3202      vivienda=False  servicios_hogar=False  condiciones_vida=True
P3203      vivienda=False  servicios_hogar=False  condiciones_vida=True
P3516      vivienda=False  servicios_hogar=False  condiciones_vida=True
P1892      vivienda=False  servicios_hogar=True  condiciones_vida=False
P5046S1    vivienda=False  servicios_hogar=True  condiciones_vida=False
P5012      vivienda=False  servicios_hogar=True  condiciones_vid